## Import dependencies

This cell imports the Python libraries used throughout the notebook for configuration, data loading, model training, evaluation, checkpointing, and experiment tracking.

In [1]:
from accelerate import Accelerator, notebook_launcher
from accelerate.utils import set_seed
from contextlib import nullcontext
from dataclasses import dataclass, asdict
from evaluate import load as load_metric
from itertools import islice
from torch.optim import AdamW
from torch.utils.data import DataLoader, Dataset
from transformers import (
    AutoModelForSeq2SeqLM,
    AutoTokenizer,
    get_linear_schedule_with_warmup,
)
import json
import math
import mlflow
import mlflow.pytorch
import numpy as np
import os
import psutil
import random
import subprocess
import shutil
import tempfile
import time
import torch


/opt/venv/lib/python3.11/site-packages/torch/cuda/__init__.py:61: FutureWarning: The pynvml package is deprecated. Please install nvidia-ml-py instead. If you did not install pynvml directly, please report this to the maintainers of the package that installed pynvml for you.
  import pynvml  # type: ignore[import]


## Check GPU availability

This cell runs `nvidia-smi` so we can confirm the notebook sees the GPU and inspect available CUDA hardware before training.

In [2]:
!nvidia-smi

Thu Apr  2 05:02:31 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 560.35.05              Driver Version: 560.35.05      CUDA Version: 12.6     |
|-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla P100-PCIE-16GB           Off |   00000000:03:00.0 Off |                    0 |
| N/A   38C    P0             26W /  250W |       0MiB /  16384MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

## Define training configuration

This cell declares the `TrainingConfig` dataclass and builds the main configuration object that controls model choice, token lengths, optimisation settings, logging, and checkpoint selection.

In [3]:
@dataclass
class TrainingConfig:
    # ── Model ──────────────────────────────────────────────
    model_name: str      = "t5-base"       # or "t5-large", "google/flan-t5-base"
    task_prefix: str     = "fix recipe: "  # prepended to every corrupted input

    # ── Tokenisation ───────────────────────────────────────
    max_input_length: int  = 512
    max_target_length: int = 512

    # ── Training ───────────────────────────────────────────
    learning_rate: float   = 3e-4
    weight_decay: float    = 0.01
    num_train_epochs: int  = 3
    per_device_train_batch_size: int = 8
    per_device_eval_batch_size: int  = 8
    gradient_accumulation_steps: int = 4   # effective batch = 8 × 4 = 32
    warmup_ratio: float    = 0.06          # fraction of total steps used for warmup
    max_grad_norm: float   = 1.0

    # ── Checkpointing ──────────────────────────────────────
    save_steps: int        = 500
    eval_steps: int        = 500
    logging_steps: int     = 50
    load_best_model_at_end: bool = True
    metric_for_best_model: str   = "eval_rougeL"
    greater_is_better: bool      = True

    # ── Reproducibility ────────────────────────────────────
    seed: int = 42

cfg = TrainingConfig()
print(asdict(cfg))

{'model_name': 't5-base', 'task_prefix': 'fix recipe: ', 'max_input_length': 512, 'max_target_length': 512, 'learning_rate': 0.0003, 'weight_decay': 0.01, 'num_train_epochs': 3, 'per_device_train_batch_size': 8, 'per_device_eval_batch_size': 8, 'gradient_accumulation_steps': 4, 'warmup_ratio': 0.06, 'max_grad_norm': 1.0, 'save_steps': 500, 'eval_steps': 500, 'logging_steps': 50, 'load_best_model_at_end': True, 'metric_for_best_model': 'eval_rougeL', 'greater_is_better': True, 'seed': 42}


## Load the tokenizer and sanity-check tokenisation

This cell loads the tokenizer for the configured model and runs a quick encode/decode example to verify the text-cleaning task is being represented correctly before we build datasets.

In [4]:
tokenizer = AutoTokenizer.from_pretrained(cfg.model_name)

# ── Quick sanity check ──────────────────────────────────────────────────────
sample_corrupted = "Ingredientss: 2 cup flur, 1 eg, salt. Diretions: mix al togeter."
sample_clean     = "Ingredients: 2 cups flour, 1 egg, salt. Directions: mix all together."

enc = tokenizer(
    cfg.task_prefix + sample_corrupted,
    max_length=cfg.max_input_length,
    truncation=True,
    return_tensors="pt"
)
dec = tokenizer(sample_clean, max_length=cfg.max_target_length, truncation=True)

print(f"Tokenizer:        {cfg.model_name}")
print(f"Vocab size:       {tokenizer.vocab_size:,}")
print(f"Input token len:  {enc['input_ids'].shape[1]}")
print(f"Target token len: {len(dec['input_ids'])}")

Tokenizer:        t5-base
Vocab size:       32,100
Input token len:  30
Target token len: 20


## Build a small synthetic dataset

This cell seeds randomness, creates mock corrupted-to-clean recipe examples, splits them into train and validation sets, and wraps them in dataset objects for notebook experimentation.

In [5]:
set_seed(cfg.seed)

# ── Minimal synthetic examples ──────────────────────────────────────────────
CORRUPTIONS = [
    ("Ingredientss: 2 cup flur, 1 eg, salt tt taste.",
     "Ingredients: 2 cups flour, 1 egg, salt to taste."),
    ("Directons: Pre-heat ovn to 180C. Bake fr 30 min.",
     "Directions: Preheat oven to 180°C. Bake for 30 minutes."),
    ("Title Choclate Cake\nIngredents sugar buttr eggs flowur",
     "Title: Chocolate Cake\nIngredients: sugar, butter, eggs, flour"),
    ("Sevings: 4\nPrep tme 15 mins\nCok time: 30minut",
     "Servings: 4\nPrep time: 15 mins\nCook time: 30 minutes"),
    ("1/2 cupp milk or creme\n2 tsp vanil extrat",
     "1/2 cup milk or cream\n2 tsp vanilla extract"),
]

def make_split(pairs, n):
    """Repeat and shuffle pairs to reach n examples."""
    pool = pairs * (n // len(pairs) + 1)
    random.shuffle(pool)
    return pool[:n]

class MockRecipeDataset(Dataset):
    def __init__(self, pairs, tokenizer, cfg):
        self.tokenizer = tokenizer
        self.cfg       = cfg
        self.inputs    = [cfg.task_prefix + c for c, _ in pairs]
        self.targets   = [t for _, t in pairs]

    def __len__(self):
        return len(self.inputs)

    def __getitem__(self, idx):
        model_inputs = self.tokenizer(
            self.inputs[idx],
            max_length=self.cfg.max_input_length,
            padding="max_length",
            truncation=True,
            return_tensors="pt",
        )
        labels = self.tokenizer(
            self.targets[idx],
            max_length=self.cfg.max_target_length,
            padding="max_length",
            truncation=True,
            return_tensors="pt",
        ).input_ids

        # Replace padding token id with -100 so it's ignored in the loss
        labels[labels == self.tokenizer.pad_token_id] = -100

        return {
            "input_ids":      model_inputs.input_ids.squeeze(0),
            "attention_mask": model_inputs.attention_mask.squeeze(0),
            "labels":         labels.squeeze(0),
        }

mock_train_dataset = MockRecipeDataset(make_split(CORRUPTIONS, 200), tokenizer, cfg)
mock_val_dataset   = MockRecipeDataset(make_split(CORRUPTIONS,  40), tokenizer, cfg)

print(f"Train examples: {len(mock_train_dataset)}")
print(f"Val examples:   {len(mock_val_dataset)}")
print(f"\nSample input_ids shape:  {mock_train_dataset[0]['input_ids'].shape}")
print(f"Sample labels shape:     {mock_train_dataset[0]['labels'].shape}")

Train examples: 200
Val examples:   40

Sample input_ids shape:  torch.Size([512])
Sample labels shape:     torch.Size([512])


## Load the model and optimiser components

This cell selects the compute device, loads the seq2seq model onto it, and prepares the optimiser and learning-rate scheduler needed for training.

In [6]:
cuda_built = torch.backends.cuda.is_built()
gpu_query = subprocess.run(
    ["nvidia-smi", "--query-gpu=name", "--format=csv,noheader"],
    capture_output=True,
    text=True,
    check=False,
) if shutil.which("nvidia-smi") else None
gpu_names = []
if gpu_query and gpu_query.returncode == 0:
    gpu_names = [line.strip() for line in gpu_query.stdout.splitlines() if line.strip()]

device_count = len(gpu_names)
cuda_runtime_visible = device_count > 0
training_device = torch.device("cuda" if cuda_built and device_count == 1 else "cpu")
launcher_preview_mode = cuda_built and device_count > 1
preview_device = torch.device("cpu") if launcher_preview_mode else training_device

print(f"Torch CUDA build available: {cuda_built}")
print(f"CUDA visible via nvidia-smi: {cuda_runtime_visible}")
print(f"CUDA device count:           {device_count}")

if device_count > 0 and not cuda_built:
    raise RuntimeError(
        "GPUs are visible via nvidia-smi, but this notebook environment has a CPU-only PyTorch build. "
        "Install a CUDA-enabled torch build before attempting GPU training."
    )

device = preview_device
print(f"Preview device: {device}")
if launcher_preview_mode:
    print(f"GPUs:   {device_count} CUDA devices detected")
    print(
        "Preview cells will stay on CPU so notebook_launcher can fork clean GPU workers later."
    )
else:
    print(f"GPUs:   {gpu_names}")

mixed_precision = "fp16" if cuda_built and device_count > 0 else "no"

class NotebookPreviewAccelerator:
    """Small single-process shim so preview cells work without initializing Accelerate state."""

    def __init__(self, device: torch.device, gradient_accumulation_steps: int):
        self.device = device
        self.gradient_accumulation_steps = gradient_accumulation_steps
        self.num_processes = 1
        self.is_main_process = True
        self.sync_gradients = True

    def unwrap_model(self, model: torch.nn.Module) -> torch.nn.Module:
        return model

    def prepare(self, *objects):
        prepared = []
        for obj in objects:
            if isinstance(obj, torch.nn.Module):
                prepared.append(obj.to(self.device))
            else:
                prepared.append(obj)
        return tuple(prepared)

    def reduce(self, tensor: torch.Tensor, reduction: str = "mean") -> torch.Tensor:
        return tensor.float().mean() if reduction == "mean" and tensor.ndim > 0 else tensor

    def pad_across_processes(self, tensor: torch.Tensor, dim: int = 0, pad_index: int = 0) -> torch.Tensor:
        return tensor

    def gather_for_metrics(self, tensors):
        return tensors

    def wait_for_everyone(self):
        return None

    def save(self, obj, path):
        torch.save(obj, path)

    def accumulate(self, model: torch.nn.Module):
        return nullcontext()

    def autocast(self):
        return nullcontext()

    def backward(self, loss: torch.Tensor):
        loss.backward()

    def clip_grad_norm_(self, parameters, max_norm: float):
        return torch.nn.utils.clip_grad_norm_(parameters, max_norm)

accelerator = NotebookPreviewAccelerator(
    device=preview_device,
    gradient_accumulation_steps=cfg.gradient_accumulation_steps,
)
is_distributed = accelerator.num_processes > 1

print(f"Accelerator device:        {accelerator.device}")
print(f"Accelerator processes:     {accelerator.num_processes}")
if device_count > 1 and not is_distributed:
    print(
        "Multiple GPUs are visible, but this notebook kernel is only a preview process. "
        "Use the notebook_launcher training cell for true distributed training."
    )

model = AutoModelForSeq2SeqLM.from_pretrained(cfg.model_name)

def unwrap_model(model: torch.nn.Module) -> torch.nn.Module:
    """Return the underlying model whether or not Accelerate wrapped it."""
    return accelerator.unwrap_model(model)

def reduce_loss(loss: torch.Tensor) -> torch.Tensor:
    """Keep loss handling safe across single-process and distributed runs."""
    return loss.float().mean() if loss.ndim > 0 else loss.float()

def generate_safely(model: torch.nn.Module, **generate_kwargs) -> torch.Tensor:
    """Generate with the unwrapped HF model so single- and multi-GPU runs share one path."""
    return unwrap_model(model).generate(**generate_kwargs)

parallel_mode = (
    "distributed"
    if is_distributed
    else ("launcher_required" if device_count > 1 and cuda_built else ("single_gpu" if device_count == 1 and cuda_built else "cpu"))
)
print(f"Parallel mode: {parallel_mode}")

# ── Parameter budget ────────────────────────────────────────────────────────
total_params     = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

print(f"\nModel:             {cfg.model_name}")
print(f"Total params:      {total_params:,}")
print(f"Trainable params:  {trainable_params:,}")
print(f"Non-trainable:     {total_params - trainable_params:,}")

Torch CUDA build available: True
CUDA visible via nvidia-smi: True
CUDA device count:           2
Preview device: cpu
GPUs:   2 CUDA devices detected
Preview cells will stay on CPU so notebook_launcher can fork clean GPU workers later.
Accelerator device:        cpu
Accelerator processes:     1
Multiple GPUs are visible, but this notebook kernel is only a preview process. Use the notebook_launcher training cell for true distributed training.


Loading weights:   0%|          | 0/257 [00:00<?, ?it/s]

Parallel mode: launcher_required

Model:             t5-base
Total params:      222,903,552
Trainable params:  222,903,552
Non-trainable:     0


## Create dataloaders

This cell constructs the training and validation dataloaders so batches are prepared with the configured batch size and ready for the training loop.

In [7]:
# ── DataLoaders ─────────────────────────────────────────────────────────────
train_loader = DataLoader(
    mock_train_dataset,
    batch_size=cfg.per_device_train_batch_size,
    shuffle=True,
    num_workers=0,      # NOTE: Change for Real Data
    pin_memory=accelerator.device.type == "cuda",
)
val_loader = DataLoader(
    mock_val_dataset,
    batch_size=cfg.per_device_eval_batch_size, 
    shuffle=False,
    num_workers=0,
    pin_memory=accelerator.device.type == "cuda",
)

# ── Optimizer ───────────────────────────────────────────────────────────────
# Separate weight decay: don't apply it to biases or LayerNorm params
no_decay = {"bias", "LayerNorm.weight"}
param_groups = [
    {
        "params": [p for n, p in unwrap_model(model).named_parameters()
                   if not any(nd in n for nd in no_decay)],
        "weight_decay": cfg.weight_decay,
    },
    {
        "params": [p for n, p in unwrap_model(model).named_parameters()
                   if any(nd in n for nd in no_decay)],
        "weight_decay": 0.0,
    },
]
optimizer = AdamW(param_groups, lr=cfg.learning_rate)

# ── LR Scheduler ────────────────────────────────────────────────────────────
steps_per_epoch   = math.ceil(len(train_loader) / cfg.gradient_accumulation_steps)
total_steps       = steps_per_epoch * cfg.num_train_epochs
warmup_steps      = int(total_steps * cfg.warmup_ratio)

scheduler = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps=warmup_steps,
    num_training_steps=total_steps,
)

model, optimizer, train_loader, val_loader, scheduler = accelerator.prepare(
    model, optimizer, train_loader, val_loader, scheduler
)

print(f"Steps per epoch:   {steps_per_epoch}")
print(f"Total opt. steps:  {total_steps}")
print(f"Warmup steps:      {warmup_steps}")
print(f"Peak LR:           {cfg.learning_rate}")

Steps per epoch:   7
Total opt. steps:  21
Warmup steps:      1
Peak LR:           0.0003


## Start an MLflow run

This cell configures the MLflow tracking server and experiment name, then opens a fresh run so training metrics, parameters, and artifacts are recorded.

In [8]:
run_id = None
EXPERIMENT_NAME     = "recipe-correction-t5"
MLFLOW_TRACKING_URI = "http://mlflow:5000"

if accelerator.is_main_process:
    mlflow.end_run()
    mlflow.set_tracking_uri(MLFLOW_TRACKING_URI)
    mlflow.set_experiment(EXPERIMENT_NAME)

    run = mlflow.start_run(run_name=f"{cfg.model_name}__lr{cfg.learning_rate}__ep{cfg.num_train_epochs}")
    run_id = run.info.run_id
    print(f"MLflow run started: {run_id}")

    # ── Log all hyperparameters ──────────────────────────────────────────────
    mlflow.log_params(asdict(cfg))

    # ── Log derived schedule values ──────────────────────────────────────────
    mlflow.log_params({
        "effective_batch_size_per_process": cfg.per_device_train_batch_size * cfg.gradient_accumulation_steps,
        "global_effective_batch_size": (
            cfg.per_device_train_batch_size * cfg.gradient_accumulation_steps * accelerator.num_processes
        ),
        "total_optimizer_steps": total_steps,
        "warmup_steps": warmup_steps,
        "num_train_examples": len(mock_train_dataset),
        "num_val_examples": len(mock_val_dataset),
        "device": str(device),
        "accelerator_processes": accelerator.num_processes,
        "trainable_params": trainable_params,
    })

    # ── Log model graph as a text artifact (optional but useful) ─────────────
    with tempfile.NamedTemporaryFile("w", suffix=".txt", delete=False) as f:
        f.write(str(unwrap_model(model)))
        tmp_path = f.name
    mlflow.log_artifact(tmp_path, artifact_path="model_summary")
    os.unlink(tmp_path)

    print("\nLogged to MLflow:")
    print(f"  Experiment : {EXPERIMENT_NAME}")
    print(f"  Run ID     : {run_id}")
    print("\nModel setup complete — ready to train.")

# NOTE: do NOT call mlflow.end_run() here.

MLflow run started: bfc5f85733e64e9cb7648c4fb8ba6292

Logged to MLflow:
  Experiment : recipe-correction-t5
  Run ID     : bfc5f85733e64e9cb7648c4fb8ba6292

Model setup complete — ready to train.


## Debug evaluation memory usage

This cell defines a more verbose evaluation helper that prints RAM usage batch by batch, which is useful for debugging memory issues during generation and metric computation.

In [9]:
def get_ram_usage():
    process = psutil.Process(os.getpid())
    return process.memory_info().rss / 1024**2  # MB

@torch.no_grad()
def evaluate_epoch(model, loader, device, cfg):
    model.eval()
    total_loss = 0.0
    num_batches = 0
    rouge_metric = load_metric("rouge") if accelerator.is_main_process else None
    
    print(f"Starting eval. Initial RAM: {get_ram_usage():.2f}MB")
    
    for i, batch in enumerate(loader):
        num_batches += 1
        print(f"--- Batch {i} ---")
        
        # Move to device
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels = batch["labels"].to(device)
        
        # Loss calculation
        outputs = model(input_ids=input_ids, attention_mask=attention_mask, labels=labels)
        loss = reduce_loss(outputs.loss)
        total_loss += accelerator.reduce(loss.detach(), reduction="mean").item()
        
        # Generation - REDUCE max_new_tokens if it still crashes
        print(f"  Generating... (RAM: {get_ram_usage():.2f}MB)")
        generated = generate_safely(
            model,
            input_ids=input_ids,
            attention_mask=attention_mask,
            max_new_tokens=20, # Hardcoded small for test
        )

        # Decode & Add
        labels_for_decode = labels.clone()
        labels_for_decode[labels_for_decode == -100] = tokenizer.pad_token_id
        
        generated = accelerator.pad_across_processes(generated, dim=1, pad_index=tokenizer.pad_token_id)
        labels_for_decode = accelerator.pad_across_processes(
            labels_for_decode, dim=1, pad_index=tokenizer.pad_token_id
        )
        gathered_generated, gathered_labels = accelerator.gather_for_metrics(
            (generated, labels_for_decode)
        )
        if accelerator.is_main_process:
            rouge_metric.add_batch(
                predictions=tokenizer.batch_decode(gathered_generated, skip_special_tokens=True),
                references=tokenizer.batch_decode(gathered_labels, skip_special_tokens=True),
            )

        # AGGRESSIVE CLEANUP
        del outputs, generated, batch, input_ids, attention_mask, labels, labels_for_decode
        if accelerator.device.type == "cuda":
            torch.cuda.empty_cache()
        
    print(f"Finalizing scores... (RAM: {get_ram_usage():.2f}MB)")
    print(f"Processed {num_batches} batch(es)")
    return (
        rouge_metric.compute(use_stemmer=True)
        if accelerator.is_main_process
        else {"rouge1": 0.0, "rouge2": 0.0, "rougeL": 0.0}
    )

# Test on ONE SINGLE BATCH
print("Testing on ONE batch...")
try:
    # We use a manual next(iter()) to be absolutely sure we don't loop
    single_batch = [next(iter(val_loader))]
    result = evaluate_epoch(model, single_batch, accelerator.device, cfg)
    print("Success!", result)
except Exception as e:
    print("Caught error:", e)

Testing on ONE batch...
Starting eval. Initial RAM: 848.58MB
--- Batch 0 ---
  Generating... (RAM: 3215.59MB)
Finalizing scores... (RAM: 2921.57MB)
Processed 1 batch(es)
Success! {'rouge1': np.float64(0.13636363636363635), 'rouge2': np.float64(0.075), 'rougeL': np.float64(0.13636363636363635), 'rougeLsum': np.float64(0.13636363636363635)}


## Define the evaluation helper

This cell builds the main evaluation routine, including ROUGE scoring and a short smoke test, so we can validate the model on the validation loader without storing every prediction in memory.

In [10]:
def decode_batch(token_ids: torch.Tensor) -> list[str]:
    """Convert a batch of token id tensors to strings, skipping special tokens."""
    return tokenizer.batch_decode(token_ids, skip_special_tokens=True)

@torch.no_grad()
def evaluate_epoch(model, loader, device, cfg) -> dict:
    """
    Run one full pass over `loader`.
    Returns a dict of scalar metrics ready for mlflow.log_metrics().
    """
    model.eval()
    total_loss = 0.0
    num_batches = 0
    rouge_metric = load_metric("rouge") if accelerator.is_main_process else None

    # 1. We iterate over the loader and add batches directly to ROUGE 
    # to avoid blowing up system RAM with a massive list of strings.
    for batch in loader:
        num_batches += 1
        input_ids      = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels         = batch["labels"].to(device)

        outputs = model(
            input_ids=input_ids,
            attention_mask=attention_mask,
            labels=labels,
        )
        loss = reduce_loss(outputs.loss)
        total_loss += accelerator.reduce(loss.detach(), reduction="mean").item()

        # Greedy decode for ROUGE (cheap; swap for beam search in final eval)
        generated = generate_safely(
            model,
            input_ids=input_ids,
            attention_mask=attention_mask,
            max_new_tokens=cfg.max_target_length,
        )

        # Replace -100 (loss-mask) back to pad_token_id before decoding
        labels_for_decode = labels.clone()
        labels_for_decode[labels_for_decode == -100] = tokenizer.pad_token_id

        generated = accelerator.pad_across_processes(generated, dim=1, pad_index=tokenizer.pad_token_id)
        labels_for_decode = accelerator.pad_across_processes(
            labels_for_decode, dim=1, pad_index=tokenizer.pad_token_id
        )
        gathered_generated, gathered_labels = accelerator.gather_for_metrics(
            (generated, labels_for_decode)
        )

        if accelerator.is_main_process:
            preds = decode_batch(gathered_generated)
            targets = decode_batch(gathered_labels)

            # Stream predictions directly to the metric buffer
            rouge_metric.add_batch(predictions=preds, references=targets)
        
        # 2. Free up heavy tensors from VRAM manually
        del generated, outputs, labels_for_decode

    avg_loss = total_loss / num_batches

    # 3. Compute everything from the accumulated buffer
    rouge_scores = (
        rouge_metric.compute(use_stemmer=True)
        if accelerator.is_main_process
        else {"rouge1": 0.0, "rouge2": 0.0, "rougeL": 0.0}
    )

    return {
        "eval_loss":    avg_loss,
        "eval_rouge1":  rouge_scores["rouge1"],
        "eval_rouge2":  rouge_scores["rouge2"],
        "eval_rougeL":  rouge_scores["rougeL"],
    }


# ── Smoke-test the eval function before training ─────────────────────────────
print("Smoke-testing evaluate_epoch() on a small slice of val set...")

# 4. Keep the smoke test lighter in launcher preview mode so it stays responsive on CPU.
smoke_test_batches = 1 if parallel_mode == "launcher_required" else 3
val_smoke_loader = islice(val_loader, smoke_test_batches)

smoke = evaluate_epoch(model, val_smoke_loader, accelerator.device, cfg)
for k, v in smoke.items():
    print(f"  {k}: {v:.4f}")
print("OK — eval utility is working.\n")

Smoke-testing evaluate_epoch() on a small slice of val set...
  eval_loss: 3.8635
  eval_rouge1: 0.0423
  eval_rouge2: 0.0217
  eval_rougeL: 0.0423
OK — eval utility is working.



## Set up checkpoint saving

This cell creates the checkpoint directory, tracks the current best validation score, and defines a helper to save model weights, tokenizer files, and per-epoch metrics.

In [11]:
CHECKPOINT_DIR = "../checkpoints/recipe-correction-tests"
os.makedirs(CHECKPOINT_DIR, exist_ok=True)

best_metric   = -float("inf")   # higher eval_rougeL = better
best_ckpt_dir = None

def save_checkpoint(model, tokenizer, epoch: int, metrics: dict) -> str:
    """
    Saves model + tokenizer to a versioned directory.
    Returns the path so it can be logged to MLflow.
    """
    ckpt_path = os.path.join(CHECKPOINT_DIR, f"epoch-{epoch:02d}")
    accelerator.wait_for_everyone()
    if accelerator.is_main_process:
        os.makedirs(ckpt_path, exist_ok=True)
        unwrap_model(model).save_pretrained(
            ckpt_path,
            is_main_process=True,
            save_function=accelerator.save,
        )
        tokenizer.save_pretrained(ckpt_path)

        # Write a small sidecar with the metrics for this checkpoint
        meta_path = os.path.join(ckpt_path, "metrics.json")
        with open(meta_path, "w") as f:
            json.dump(metrics, f, indent=2)
    accelerator.wait_for_everyone()

    return ckpt_path

## Train and validate the model

This cell runs the full training loop with mixed precision, gradient accumulation, periodic MLflow logging, end-of-epoch validation, and best-checkpoint tracking.

In [12]:
RUN_SUMMARY_PATH = os.path.join(CHECKPOINT_DIR, "last_run_summary.json")

def notebook_training_worker(cfg_dict: dict):
    cfg_local = TrainingConfig(**cfg_dict)
    accelerator_local = Accelerator(
        mixed_precision=("fp16" if torch.cuda.is_available() else "no"),
        gradient_accumulation_steps=cfg_local.gradient_accumulation_steps,
    )
    device_local = accelerator_local.device
    set_seed(cfg_local.seed)

    tokenizer_local = AutoTokenizer.from_pretrained(cfg_local.model_name)
    train_dataset_local = MockRecipeDataset(make_split(CORRUPTIONS, 200), tokenizer_local, cfg_local)
    val_dataset_local   = MockRecipeDataset(make_split(CORRUPTIONS,  40), tokenizer_local, cfg_local)

    train_loader_local = DataLoader(
        train_dataset_local,
        batch_size=cfg_local.per_device_train_batch_size,
        shuffle=True,
        num_workers=0,
        pin_memory=device_local.type == "cuda",
    )
    val_loader_local = DataLoader(
        val_dataset_local,
        batch_size=cfg_local.per_device_eval_batch_size,
        shuffle=False,
        num_workers=0,
        pin_memory=device_local.type == "cuda",
    )

    model_local = AutoModelForSeq2SeqLM.from_pretrained(cfg_local.model_name)
    no_decay = {"bias", "LayerNorm.weight"}
    param_groups_local = [
        {
            "params": [
                p for n, p in model_local.named_parameters()
                if not any(nd in n for nd in no_decay)
            ],
            "weight_decay": cfg_local.weight_decay,
        },
        {
            "params": [
                p for n, p in model_local.named_parameters()
                if any(nd in n for nd in no_decay)
            ],
            "weight_decay": 0.0,
        },
    ]
    optimizer_local = AdamW(param_groups_local, lr=cfg_local.learning_rate)
    steps_per_epoch_local = math.ceil(len(train_loader_local) / cfg_local.gradient_accumulation_steps)
    total_steps_local = steps_per_epoch_local * cfg_local.num_train_epochs
    warmup_steps_local = int(total_steps_local * cfg_local.warmup_ratio)
    scheduler_local = get_linear_schedule_with_warmup(
        optimizer_local,
        num_warmup_steps=warmup_steps_local,
        num_training_steps=total_steps_local,
    )

    model_local, optimizer_local, train_loader_local, val_loader_local, scheduler_local = accelerator_local.prepare(
        model_local, optimizer_local, train_loader_local, val_loader_local, scheduler_local
    )

    def unwrap_model_local(model: torch.nn.Module) -> torch.nn.Module:
        return accelerator_local.unwrap_model(model)

    def reduce_loss_local(loss: torch.Tensor) -> torch.Tensor:
        return loss.float().mean() if loss.ndim > 0 else loss.float()

    def generate_safely_local(model: torch.nn.Module, **generate_kwargs) -> torch.Tensor:
        return unwrap_model_local(model).generate(**generate_kwargs)

    def decode_batch_local(token_ids: torch.Tensor) -> list[str]:
        return tokenizer_local.batch_decode(token_ids, skip_special_tokens=True)

    @torch.no_grad()
    def evaluate_epoch_local(model, loader) -> dict:
        model.eval()
        total_loss_local = 0.0
        num_batches_local = 0
        rouge_metric_local = load_metric("rouge") if accelerator_local.is_main_process else None

        for batch in loader:
            num_batches_local += 1
            input_ids = batch["input_ids"].to(device_local)
            attention_mask = batch["attention_mask"].to(device_local)
            labels = batch["labels"].to(device_local)

            outputs = model(
                input_ids=input_ids,
                attention_mask=attention_mask,
                labels=labels,
            )
            loss = reduce_loss_local(outputs.loss)
            total_loss_local += accelerator_local.reduce(loss.detach(), reduction="mean").item()

            generated = generate_safely_local(
                model,
                input_ids=input_ids,
                attention_mask=attention_mask,
                max_new_tokens=cfg_local.max_target_length,
            )

            labels_for_decode = labels.clone()
            labels_for_decode[labels_for_decode == -100] = tokenizer_local.pad_token_id
            generated = accelerator_local.pad_across_processes(generated, dim=1, pad_index=tokenizer_local.pad_token_id)
            labels_for_decode = accelerator_local.pad_across_processes(
                labels_for_decode, dim=1, pad_index=tokenizer_local.pad_token_id
            )
            gathered_generated, gathered_labels = accelerator_local.gather_for_metrics(
                (generated, labels_for_decode)
            )

            if accelerator_local.is_main_process:
                preds = decode_batch_local(gathered_generated)
                targets = decode_batch_local(gathered_labels)
                rouge_metric_local.add_batch(predictions=preds, references=targets)

        avg_loss_local = total_loss_local / max(num_batches_local, 1)
        rouge_scores_local = (
            rouge_metric_local.compute(use_stemmer=True)
            if accelerator_local.is_main_process
            else {"rouge1": 0.0, "rouge2": 0.0, "rougeL": 0.0}
        )
        return {
            "eval_loss": avg_loss_local,
            "eval_rouge1": rouge_scores_local["rouge1"],
            "eval_rouge2": rouge_scores_local["rouge2"],
            "eval_rougeL": rouge_scores_local["rougeL"],
        }

    def save_checkpoint_local(model, epoch: int, metrics: dict) -> str:
        ckpt_path_local = os.path.join(CHECKPOINT_DIR, f"epoch-{epoch:02d}")
        accelerator_local.wait_for_everyone()
        if accelerator_local.is_main_process:
            os.makedirs(ckpt_path_local, exist_ok=True)
            unwrap_model_local(model).save_pretrained(
                ckpt_path_local,
                is_main_process=True,
                save_function=accelerator_local.save,
            )
            tokenizer_local.save_pretrained(ckpt_path_local)
            with open(os.path.join(ckpt_path_local, "metrics.json"), "w") as f:
                json.dump(metrics, f, indent=2)
        accelerator_local.wait_for_everyone()
        return ckpt_path_local

    best_metric_local = -float("inf")
    best_ckpt_dir_local = None
    global_step_local = 0
    train_loss_accum_local = 0.0
    optimizer_local.zero_grad()
    run_id_local = None

    if accelerator_local.is_main_process:
        mlflow.end_run()
        mlflow.set_tracking_uri(MLFLOW_TRACKING_URI)
        mlflow.set_experiment(EXPERIMENT_NAME)
        run = mlflow.start_run(run_name=f"{cfg_local.model_name}__lr{cfg_local.learning_rate}__ep{cfg_local.num_train_epochs}")
        run_id_local = run.info.run_id
        mlflow.log_params(asdict(cfg_local))
        mlflow.log_params({
            "effective_batch_size_per_process": cfg_local.per_device_train_batch_size * cfg_local.gradient_accumulation_steps,
            "global_effective_batch_size": cfg_local.per_device_train_batch_size * cfg_local.gradient_accumulation_steps * accelerator_local.num_processes,
            "total_optimizer_steps": total_steps_local,
            "warmup_steps": warmup_steps_local,
            "num_train_examples": len(train_dataset_local),
            "num_val_examples": len(val_dataset_local),
            "device": str(device_local),
            "accelerator_processes": accelerator_local.num_processes,
            "parallel_mode": "distributed" if accelerator_local.num_processes > 1 else "single_gpu",
        })
        print(
            f"Starting training via notebook_launcher with {accelerator_local.num_processes} process(es)."
        )

    for epoch in range(1, cfg_local.num_train_epochs + 1):
        model_local.train()
        epoch_start_local = time.time()

        for batch in train_loader_local:
            input_ids = batch["input_ids"].to(device_local)
            attention_mask = batch["attention_mask"].to(device_local)
            labels = batch["labels"].to(device_local)

            with accelerator_local.accumulate(model_local):
                with accelerator_local.autocast():
                    outputs = model_local(
                        input_ids=input_ids,
                        attention_mask=attention_mask,
                        labels=labels,
                    )
                    loss = reduce_loss_local(outputs.loss)

                train_loss_accum_local += accelerator_local.reduce(loss.detach(), reduction="mean").item()
                accelerator_local.backward(loss)

                if accelerator_local.sync_gradients:
                    accelerator_local.clip_grad_norm_(model_local.parameters(), cfg_local.max_grad_norm)

                optimizer_local.step()
                scheduler_local.step()
                optimizer_local.zero_grad()

            if accelerator_local.sync_gradients:
                global_step_local += 1
                if accelerator_local.is_main_process and global_step_local % cfg_local.logging_steps == 0:
                    avg_train_loss_local = train_loss_accum_local / cfg_local.logging_steps
                    current_lr_local = scheduler_local.get_last_lr()[0]
                    mlflow.log_metrics(
                        {"train_loss": avg_train_loss_local, "learning_rate": current_lr_local},
                        step=global_step_local,
                    )
                    print(
                        f"  Epoch {epoch}/{cfg_local.num_train_epochs} | step {global_step_local:>5} | train_loss {avg_train_loss_local:.4f} | lr {current_lr_local:.2e}"
                    )
                    train_loss_accum_local = 0.0

        epoch_elapsed_local = time.time() - epoch_start_local
        eval_metrics_local = evaluate_epoch_local(model_local, val_loader_local)
        accelerator_local.wait_for_everyone()
        ckpt_path_local = save_checkpoint_local(model_local, epoch, eval_metrics_local)

        if accelerator_local.is_main_process:
            eval_metrics_local["epoch"] = epoch
            eval_metrics_local["epoch_time_sec"] = round(epoch_elapsed_local, 2)
            eval_metrics_local["samples_per_sec"] = round(len(train_dataset_local) / epoch_elapsed_local, 1)
            mlflow.log_metrics(eval_metrics_local, step=global_step_local)
            mlflow.log_artifacts(ckpt_path_local, artifact_path=f"checkpoints/epoch-{epoch:02d}")

            current_metric_local = eval_metrics_local[cfg_local.metric_for_best_model]
            if current_metric_local > best_metric_local:
                best_metric_local = current_metric_local
                best_ckpt_dir_local = ckpt_path_local
                mlflow.log_metric("best_rougeL", best_metric_local, step=global_step_local)
                mlflow.set_tag("best_checkpoint", ckpt_path_local)
                print(f"  ★ New best {cfg_local.metric_for_best_model}: {best_metric_local:.4f} → {ckpt_path_local}\n")

    if accelerator_local.is_main_process:
        if best_ckpt_dir_local:
            best_model_local = AutoModelForSeq2SeqLM.from_pretrained(best_ckpt_dir_local)
            mlflow.pytorch.log_model(
                pytorch_model=best_model_local,
                artifact_path="best-model",
                registered_model_name="recipe-correction-t5",
            )
        mlflow.set_tags({
            "model_name": cfg_local.model_name,
            "task": "recipe-correction",
            "data_source": "mock-synthetic",
            "status": "complete",
            "best_checkpoint": best_ckpt_dir_local or "none",
            f"best_{cfg_local.metric_for_best_model}": f"{best_metric_local:.4f}",
        })
        with open(RUN_SUMMARY_PATH, "w") as f:
            json.dump(
                {
                    "best_metric": best_metric_local,
                    "best_ckpt_dir": best_ckpt_dir_local,
                    "global_step": global_step_local,
                    "run_id": run_id_local,
                    "parallel_mode": "distributed" if accelerator_local.num_processes > 1 else "single_gpu",
                    "num_processes": accelerator_local.num_processes,
                },
                f,
                indent=2,
            )
        mlflow.end_run()
        print(f"Training complete. Best {cfg_local.metric_for_best_model}: {best_metric_local:.4f}")
        print(f"Best checkpoint: {best_ckpt_dir_local}")

if os.path.exists(RUN_SUMMARY_PATH):
    os.remove(RUN_SUMMARY_PATH)

requested_processes = device_count if device_count > 0 and cuda_built else 1
if requested_processes > 1:
    print(f"Launching notebook training across {requested_processes} GPU processes...")
    notebook_launcher(
        notebook_training_worker,
        (asdict(cfg),),
        num_processes=requested_processes,
    )
else:
    print("Launching notebook training in single-process mode...")
    notebook_training_worker(asdict(cfg))

if not os.path.exists(RUN_SUMMARY_PATH):
    raise FileNotFoundError(f"Training finished but did not write {RUN_SUMMARY_PATH}")

with open(RUN_SUMMARY_PATH) as f:
    training_summary = json.load(f)

best_metric = training_summary["best_metric"]
best_ckpt_dir = training_summary["best_ckpt_dir"]
global_step = training_summary["global_step"]
run_id = training_summary["run_id"]
parallel_mode = training_summary["parallel_mode"]
print(f"Notebook launcher completed with mode: {parallel_mode} ({training_summary['num_processes']} process(es))")
print(f"Best checkpoint loaded from summary: {best_ckpt_dir}")

Launching notebook training across 2 GPU processes...
Launching training on 2 CUDAs.


E0402 05:03:38.530000 1120 torch/distributed/elastic/multiprocessing/api.py:732] failed (exitcode: 1) local_rank: 0 (pid: 1281) of fn: notebook_training_worker (start_method: fork)
E0402 05:03:38.530000 1120 torch/distributed/elastic/multiprocessing/api.py:732] Traceback (most recent call last):
E0402 05:03:38.530000 1120 torch/distributed/elastic/multiprocessing/api.py:732]   File "/opt/venv/lib/python3.11/site-packages/torch/distributed/elastic/multiprocessing/api.py", line 687, in _poll
E0402 05:03:38.530000 1120 torch/distributed/elastic/multiprocessing/api.py:732]     self._pc.join(-1)
E0402 05:03:38.530000 1120 torch/distributed/elastic/multiprocessing/api.py:732]   File "/opt/venv/lib/python3.11/site-packages/torch/multiprocessing/spawn.py", line 203, in join
E0402 05:03:38.530000 1120 torch/distributed/elastic/multiprocessing/api.py:732]     raise ProcessRaisedException(msg, error_index, failed_process.pid)
E0402 05:03:38.530000 1120 torch/distributed/elastic/multiprocessing/ap

ChildFailedError: 
============================================================
notebook_training_worker FAILED
------------------------------------------------------------
Failures:
  <NO_OTHER_FAILURES>
------------------------------------------------------------
Root Cause (first observed failure):
[0]:
  time      : 2026-04-02_05:03:38
  host      : e09f736ac461
  rank      : 0 (local_rank: 0)
  exitcode  : 1 (pid: 1281)
  error_file: /tmp/torchelastic_1aszqg0e/none_auc60tq3/attempt_0/0/error.json
  traceback : Traceback (most recent call last):
    File "/opt/venv/lib/python3.11/site-packages/torch/distributed/elastic/multiprocessing/errors/__init__.py", line 355, in wrapper
      return f(*args, **kwargs)
             ^^^^^^^^^^^^^^^^^^
    File "/tmp/ipykernel_1120/1035142663.py", line 5, in notebook_training_worker
      accelerator_local = Accelerator(
                          ^^^^^^^^^^^^
    File "/opt/venv/lib/python3.11/site-packages/accelerate/accelerator.py", line 461, 

## Register the best model in MLflow

This cell reloads the best checkpoint, logs it as the canonical MLflow model artifact, adds summary tags to the run, and then closes the experiment run.

In [ ]:
print("Best-model registration and run finalization are handled inside notebook_training_worker().")
print(f"Recorded run_id: {run_id}")
print(f"Best checkpoint: {best_ckpt_dir}")

## Run inference examples

This cell loads the best saved checkpoint and generates corrected recipe text for a few sample noisy inputs so we can qualitatively inspect model behaviour.

In [ ]:
if accelerator.is_main_process:
    # Load best checkpoint (or replace path with any checkpoint dir)
    inf_model     = AutoModelForSeq2SeqLM.from_pretrained(best_ckpt_dir).to(device)
    inf_tokenizer = AutoTokenizer.from_pretrained(best_ckpt_dir)
    inf_model.eval()

    TEST_INPUTS = [
        "Ingredientss: 2 cup flur, 1 eg, salt tt taste.",
        "Directons: Pre-heat ovn to 180C. Bake fr 30 min.",
        "Title Choclate Cake\nIngredents sugar buttr eggs flowur",
    ]

    print("─" * 60)
    for raw in TEST_INPUTS:
        prompt = cfg.task_prefix + raw
        enc    = inf_tokenizer(
            prompt,
            return_tensors="pt",
            max_length=cfg.max_input_length,
            truncation=True,
        ).to(device)

        with torch.no_grad():
            out = generate_safely(
                inf_model,
                **enc,
                max_new_tokens=cfg.max_target_length,
                num_beams=4,
                early_stopping=True,
            )

        decoded = inf_tokenizer.decode(out[0], skip_special_tokens=True)
        print(f"IN  : {raw}")
        print(f"OUT : {decoded}")
        print("─" * 60)

---
## MLflow Instrumentation

Deep logging layer on top of the training loop:
- System & GPU metrics (memory, utilisation, throughput)
- Per-example prediction traces (spot-check table as an artifact)
- Learning rate curve snapshot
- Custom `MLflowCallback` — drop-in for HuggingFace `Trainer` when you migrate
- Experiment comparison helper — query the tracking server and rank runs

class SystemMetricsLogger:
    """
    Background thread that polls system and GPU metrics every `interval_sec`
    seconds and logs them to the currently active MLflow run.

    Usage:
        logger = SystemMetricsLogger(interval_sec=30)
        logger.start()
        # ... training ...
        logger.stop()
    """

    def __init__(self, interval_sec: int = 30):
        self.interval  = interval_sec
        self._stop_evt = threading.Event()
        self._thread   = threading.Thread(target=self._run, daemon=True)
        self._step     = 0
        self.has_cuda  = torch.cuda.is_available()

    def start(self):
        self._thread.start()
        print(f"SystemMetricsLogger started (every {self.interval}s, "
              f"CUDA={'yes' if self.has_cuda else 'no'})")

    def stop(self):
        self._stop_evt.set()
        self._thread.join()
        print("SystemMetricsLogger stopped.")

    def _collect(self) -> dict:
        metrics = {}

        # ── CPU / RAM ───────────────────────────────────────────────────────
        metrics["sys_cpu_percent"]    = psutil.cpu_percent(interval=None)
        mem = psutil.virtual_memory()
        metrics["sys_ram_used_gb"]    = round(mem.used  / 1e9, 2)
        metrics["sys_ram_total_gb"]   = round(mem.total / 1e9, 2)
        metrics["sys_ram_percent"]    = mem.percent

        # ── GPU (CUDA) ──────────────────────────────────────────────────────
        if self.has_cuda:
            for i in range(torch.cuda.device_count()):
                props = torch.cuda.get_device_properties(i)
                alloc = torch.cuda.memory_allocated(i)
                resvd = torch.cuda.memory_reserved(i)
                total = props.total_memory

                metrics[f"gpu{i}_mem_allocated_gb"] = round(alloc / 1e9, 3)
                metrics[f"gpu{i}_mem_reserved_gb"]  = round(resvd / 1e9, 3)
                metrics[f"gpu{i}_mem_total_gb"]     = round(total / 1e9, 3)
                metrics[f"gpu{i}_mem_util_pct"]     = round(alloc / total * 100, 1)

                # torch.cuda.utilization() requires pynvml — graceful fallback
                try:
                    metrics[f"gpu{i}_utilization_pct"] = torch.cuda.utilization(i)
                except Exception:
                    pass

        return metrics

    def _run(self):
        while not self._stop_evt.wait(self.interval):
            metrics = self._collect()
            if accelerator.is_main_process:
                try:
                    mlflow.log_metrics(metrics, step=self._step)
                except Exception as e:
                    print(f"[SystemMetricsLogger] MLflow log error: {e}")
            self._step += 1


# ── Log static environment info once as params ───────────────────────────────
def log_environment_info():
    env_info = {
        "python_version":  platform.python_version(),
        "os":              platform.system(),
        "cpu_count":       psutil.cpu_count(logical=True),
        "ram_total_gb":    round(psutil.virtual_memory().total / 1e9, 1),
        "cuda_available":  torch.cuda.is_available(),
        "accelerator_processes": accelerator.num_processes,
    }
    if torch.cuda.is_available():
        env_info["cuda_version"]   = torch.version.cuda
        env_info["gpu_name"]       = torch.cuda.get_device_name(0)
        env_info["gpu_count"]      = torch.cuda.device_count()
        env_info["gpu_total_gb"]   = round(
            torch.cuda.get_device_properties(0).total_memory / 1e9, 1
        )

    import transformers
    env_info["transformers_version"] = transformers.__version__
    env_info["torch_version"]        = torch.__version__
    env_info["mlflow_version"]       = mlflow.__version__

    if accelerator.is_main_process:
        mlflow.log_params(env_info)
        print("Environment info logged:")
        for k, v in env_info.items():
            print(f"  {k}: {v}")


# ── Smoke test (logs one sample collection without training) ─────────────────
if accelerator.is_main_process:
    _active = mlflow.active_run()
    if _active is None:
        print("No active run — opening a temp run for smoke test.")
        mlflow.start_run(run_name="smoke-test-system-metrics")
        _opened_here = True
    else:
        _opened_here = False

    log_environment_info()

    sys_logger = SystemMetricsLogger(interval_sec=10)
    sys_logger.start()
    time.sleep(12)   # let it fire once
    sys_logger.stop()

    if _opened_here:
        mlflow.end_run()

    print("\nCell ready — attach sys_logger.start() / stop() around your training loop.")

In [ ]:
def build_prediction_trace(
    model,
    tokenizer,
    dataset,
    cfg,
    device,
    n_examples: int = 20,
    artifact_subdir: str = "prediction_traces",
    step: int = 0,
) -> pd.DataFrame:
    """
    Runs inference on `n_examples` from `dataset`, returns a DataFrame
    and logs it as both a CSV and a styled HTML artifact to MLflow.
    """
    model.eval()
    records = []

    indices = list(range(min(n_examples, len(dataset))))

    for idx in indices:
        item = dataset[idx]
        input_ids      = item["input_ids"].unsqueeze(0).to(device)
        attention_mask = item["attention_mask"].unsqueeze(0).to(device)
        labels         = item["labels"].unsqueeze(0).clone()
        labels[labels == -100] = tokenizer.pad_token_id

        with torch.no_grad():
            generated = generate_safely(
                model,
                input_ids=input_ids,
                attention_mask=attention_mask,
                max_new_tokens=cfg.max_target_length,
                num_beams=4,
                early_stopping=True,
            )

        raw_input  = tokenizer.decode(input_ids[0],  skip_special_tokens=True)
        target     = tokenizer.decode(labels[0],     skip_special_tokens=True)
        prediction = tokenizer.decode(generated[0],  skip_special_tokens=True)

        # Strip the task prefix from the displayed input
        display_input = raw_input.replace(cfg.task_prefix, "", 1)

        # Simple character-level exact match
        exact_match = int(prediction.strip() == target.strip())

        records.append({
            "idx":            idx,
            "corrupted_input": display_input,
            "target":         target,
            "prediction":     prediction,
            "exact_match":    exact_match,
        })

    df = pd.DataFrame(records)

    # ── Aggregate metrics ────────────────────────────────────────────────────
    exact_match_rate = df["exact_match"].mean()
    if accelerator.is_main_process:
        mlflow.log_metric("trace_exact_match_rate", exact_match_rate, step=step)

    # ── Save CSV artifact ────────────────────────────────────────────────────
    with tempfile.TemporaryDirectory() as tmp:
        csv_path  = os.path.join(tmp, f"predictions_step{step:05d}.csv")
        html_path = os.path.join(tmp, f"predictions_step{step:05d}.html")

        df.to_csv(csv_path, index=False)

        # Styled HTML — readable in the MLflow artifact browser
        styled = (
            df.style
            .set_caption(f"Prediction trace — step {step} | exact match: {exact_match_rate:.1%}")
            .applymap(
                lambda v: "background-color: #d4edda" if v == 1
                          else "background-color: #f8d7da",
                subset=["exact_match"],
            )
            .set_properties(**{
                "text-align":   "left",
                "font-size":    "13px",
                "white-space":  "pre-wrap",
                "max-width":    "400px",
            })
        )
        styled.to_html(html_path)

        if accelerator.is_main_process:
            mlflow.log_artifact(csv_path,  artifact_path=artifact_subdir)
            mlflow.log_artifact(html_path, artifact_path=artifact_subdir)

    if accelerator.is_main_process:
        print(f"Prediction trace logged — {len(df)} examples, "
              f"exact match: {exact_match_rate:.1%}")
    return df


# ── Run a trace against the best checkpoint from §4 ─────────────────────────
if accelerator.is_main_process:
    _active = mlflow.active_run()
    if _active is None:
        mlflow.start_run(run_name="smoke-test-trace")
        _opened_here = True
    else:
        _opened_here = False

    # Use best checkpoint if available, otherwise the live model
    _infer_model = (
        AutoModelForSeq2SeqLM.from_pretrained(best_ckpt_dir).to(device)
        if "best_ckpt_dir" in dir() and best_ckpt_dir
        else model
    )

    trace_df = build_prediction_trace(
        model=_infer_model,
        tokenizer=tokenizer,
        dataset=mock_val_dataset,
        cfg=cfg,
        device=device,
        n_examples=20,
        step=global_step if "global_step" in dir() else 0,
    )

    display(trace_df)

    if _opened_here:
        mlflow.end_run()

In [ ]:
import matplotlib
matplotlib.use("Agg")   # non-interactive backend — safe inside Jupyter
import matplotlib.pyplot as plt
import os, tempfile
import torch
from transformers import get_linear_schedule_with_warmup
from torch.optim import AdamW

def log_lr_curve(cfg, total_steps: int, warmup_steps: int, step: int = 0):
    """
    Simulates the LR schedule for the full training run and logs
    the curve as a PNG artifact to the active MLflow run.
    """
    # Dummy model + optimiser — we only need the scheduler values
    _dummy_param = torch.nn.Parameter(torch.zeros(1))
    _opt         = AdamW([_dummy_param], lr=cfg.learning_rate)
    _sched       = get_linear_schedule_with_warmup(
        _opt,
        num_warmup_steps=warmup_steps,
        num_training_steps=total_steps,
    )

    lrs = []
    for _ in range(total_steps):
        lrs.append(_opt.param_groups[0]["lr"])
        _opt.step()
        _sched.step()

    fig, ax = plt.subplots(figsize=(9, 3.5))
    ax.plot(lrs, color="#4A90D9", linewidth=1.5, label="LR")
    ax.axvline(warmup_steps, color="#E25C3B", linewidth=1,
               linestyle="--", label=f"Warmup end (step {warmup_steps})")
    ax.set_xlabel("Optimiser step", fontsize=11)
    ax.set_ylabel("Learning rate", fontsize=11)
    ax.set_title(
        f"LR schedule — {cfg.model_name} | "
        f"peak {cfg.learning_rate:.0e} | warmup {warmup_steps} steps",
        fontsize=11,
    )
    ax.legend(fontsize=10)
    ax.grid(True, alpha=0.3)
    fig.tight_layout()

    with tempfile.TemporaryDirectory() as tmp:
        path = os.path.join(tmp, "lr_curve.png")
        fig.savefig(path, dpi=150)
        if accelerator.is_main_process:
            mlflow.log_artifact(path, artifact_path="plots")
            print(f"LR curve logged to MLflow (total_steps={total_steps}, "
                  f"warmup={warmup_steps}, peak_lr={cfg.learning_rate:.0e})")

    plt.close(fig)


if accelerator.is_main_process:
    _active = mlflow.active_run()
    if _active is None:
        mlflow.start_run(run_name="smoke-test-lr-curve")
        _opened_here = True
    else:
        _opened_here = False

    log_lr_curve(cfg, total_steps, warmup_steps)

    if _opened_here:
        mlflow.end_run()